<a href="https://colab.research.google.com/github/PalakTolwani/Datascience/blob/main/Bigdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile avg_age.py
from mrjob.job import MRJob
import sys

class AvgApproval(MRJob):
  def mapper(self, _, line):
    fields = line.split(',')
    if fields[1] == 'income':
      return
    age = int(fields[6])
    approval = int(fields[7])
    yield(age, approval)

  def reducer(self, key, values):
    values = list(values)
    yield(key, sum(values)/len(values))

if __name__ == "__main__":
  # Remove the -f argument added by the Colab kernel
  if "-f" in sys.argv:
    sys.argv.remove("-f")
  AvgApproval.run()

Writing avg_age.py


In [3]:
!pip install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 6.3 MB/s eta 0:00:00


In [4]:
from mrjob.job import MRJob

In [5]:
!python avg_age.py loan_data.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Traceback (most recent call last):
  File "/content/avg_age.py", line 21, in <module>
    AvgApproval.run()
  File "/usr/local/lib/python3.11/dist-packages/mrjob/job.py", line 616, in run
    cls().execute()
  File "/usr/local/lib/python3.11/dist-packages/mrjob/job.py", line 687, in execute
    self.run_job()
  File "/usr/local/lib/python3.11/dist-packages/mrjob/job.py", line 636, in run_job
    runner.run()
  File "/usr/local/lib/python3.11/dist-packages/mrjob/runner.py", line 500, in run
    self._check_input_paths()
  File "/usr/local/lib/python3.11/dist-packages/mrjob/runner.py", line 1133, in _check_input_paths
    self._check_input_path(path)
  File "/usr/local/lib/python3.11/dist-packages/mrjob/runner.py", line 1146, in _check_input_path
    raise IOError(
OSError: Input path loan_data.csv does not exist!


In [6]:
!ls


avg_age.py  sample_data


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [8]:
spark = SparkSession.builder.appName('LoanApproval').getOrCreate()

In [10]:
df = spark.read.csv('loan_data.csv', header = True, inferSchema = True)
df

DataFrame[_c0: int, income: double, credit_score: double, loan_amount: double, years_employed: int, debt_to_income: double, age: int, approved: int]

In [11]:
df.show()

+---+------------------+-----------------+------------------+--------------+------------------+---+--------+
|_c0|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|
+---+------------------+-----------------+------------------+--------------+------------------+---+--------+
|  0|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|
|  1| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|
|  2|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|
|  3| 72.84544784612038|678.1484618345286|23.530632222944263|            16| 24.32608764281806| 39|       1|
|  4| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.408692224813464| 51|       1|
|  5|46.487945645762295|625.6437308117652|33.934853854217494|            18|33.847510547633135| 62|       1|
|  6| 73.6881922326

In [12]:
df.describe

<bound method DataFrame.describe of DataFrame[_c0: int, income: double, credit_score: double, loan_amount: double, years_employed: int, debt_to_income: double, age: int, approved: int]>

In [13]:
df.drop("_c0").show()

+------------------+-----------------+------------------+--------------+------------------+---+--------+
|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|
+------------------+-----------------+------------------+--------------+------------------+---+--------+
|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|
| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|
|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|
| 72.84544784612038|678.1484618345286|23.530632222944263|            16| 24.32608764281806| 39|       1|
| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.408692224813464| 51|       1|
|46.487945645762295|625.6437308117652|33.934853854217494|            18|33.847510547633135| 62|       1|
| 73.68819223261087|620.3803037880565| 38.9519322002773

In [14]:
total = df.count()
print("Total number of rows: ", total)

Total number of rows:  500


In [15]:
len(df.columns)

8

In [16]:
Filter = df.filter(df['income']<df['loan_amount'])

In [18]:
Filter.show()

+---+------------------+-----------------+------------------+--------------+--------------------+---+--------+
|_c0|            income|     credit_score|       loan_amount|years_employed|      debt_to_income|age|approved|
+---+------------------+-----------------+------------------+--------------+--------------------+---+--------+
| 10| 43.04873460781307|663.5228412889919|43.173940656343255|            29|   17.23518191940357| 28|       0|
| 13|21.300796330133032|604.6218168979201| 23.10812181910432|            20|  22.179810692369646| 52|       0|
| 14|24.126232512304508|621.1614334715833|47.359638031652494|             3|  23.801274525771266| 29|       0|
| 19|28.815444479970626|687.5693561685895|34.241659464019165|            29|  19.471649653788205| 24|       0|
| 23|28.628777206798148| 678.529933429658| 51.42270358611864|             0|   32.94383417979177| 57|       0|
| 24|41.834259132122256|611.8370421728741|47.275431701007115|            17|  30.591592889031848| 22|       0|
|

In [17]:
Func = df.groupBy('approved').count()

In [19]:
avg = Func.withColumn('Average', col('count')/total)
avg.show()

+--------+-----+-------+
|approved|count|Average|
+--------+-----+-------+
|       1|  297|  0.594|
|       0|  203|  0.406|
+--------+-----+-------+



In [20]:
la = df.groupBy('approved').avg('loan_amount', 'income')
la.show()

+--------+------------------+------------------+
|approved|  avg(loan_amount)|       avg(income)|
+--------+------------------+------------------+
|       1|29.265765620866603| 52.98864461824399|
|       0| 33.74625743745832|45.880086245302486|
+--------+------------------+------------------+



In [21]:
filtered_df = df.filter((df['income']<df['loan_amount']) & (df['age']>20))
filtered_df.agg({'age':'avg'}).show()

+-----------------+
|         avg(age)|
+-----------------+
|45.21917808219178|
+-----------------+



#Model Building

In [22]:
df.columns

['_c0',
 'income',
 'credit_score',
 'loan_amount',
 'years_employed',
 'debt_to_income',
 'age',
 'approved']

In [23]:
assembler = VectorAssembler(inputCols=['age', 'income', 'loan_amount', 'approved'], outputCol='features')
assembled_df = assembler.transform(df)
assembled_df.show()

+---+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------+
|_c0|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|            features|
+---+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------+
|  0|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|[22.0,57.45071229...|
|  1| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|[18.0,47.92603548...|
|  2|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|[34.0,59.71532807...|
|  3| 72.84544784612038|678.1484618345286|23.530632222944263|            16| 24.32608764281806| 39|       1|[39.0,72.84544784...|
|  4| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.40869222481

In [24]:
assembled_df.select('approved','features').show(truncate = False)

+--------+------------------------------------------------+
|approved|features                                        |
+--------+------------------------------------------------+
|1       |[22.0,57.450712295168486,43.99355436586002,1.0] |
|1       |[18.0,47.92603548243223,39.246336829127685,1.0] |
|0       |[34.0,59.715328071510385,30.59630369920174,0.0] |
|1       |[39.0,72.84544784612038,23.530632222944263,1.0] |
|1       |[51.0,46.48769937914996,36.982233136135896,1.0] |
|1       |[62.0,46.487945645762295,33.934853854217494,1.0]|
|0       |[58.0,73.68819223261087,38.95193220027733,0.0]  |
|0       |[46.0,61.51152093729363,36.351718016819696,0.0] |
|1       |[21.0,42.95788421097572,40.49552715319335,1.0]  |
|0       |[22.0,58.13840065378947,24.64764788439432,0.0]  |
|0       |[28.0,43.04873460781307,43.173940656343255,0.0] |
|0       |[68.0,43.01405369644615,31.975996046923996,0.0] |
|1       |[38.0,53.629434073490515,50.75260872625265,1.0] |
|0       |[52.0,21.300796330133032,23.10

In [25]:
train_df, test_df = assembled_df.randomSplit([0.7, 0.3], seed =100)

In [27]:
train_df.show()

+---+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------+
|_c0|            income|     credit_score|       loan_amount|years_employed|    debt_to_income|age|approved|            features|
+---+------------------+-----------------+------------------+--------------+------------------+---+--------+--------------------+
|  0|57.450712295168486| 696.308877376582| 43.99355436586002|            27|21.223491386654434| 22|       1|[22.0,57.45071229...|
|  1| 47.92603548243223|745.4708320235065|39.246336829127685|            27|24.836012237442116| 18|       1|[18.0,47.92603548...|
|  2|59.715328071510385|580.0716213090429| 30.59630369920174|            18|20.943750087879337| 34|       0|[34.0,59.71532807...|
|  4| 46.48769937914996|617.4678715439087|36.982233136135896|            13|30.408692224813464| 51|       1|[51.0,46.48769937...|
|  5|46.487945645762295|625.6437308117652|33.934853854217494|            18|33.84751054763

In [28]:
train_df.count()

346

In [26]:
test_df.count()

154

#Random Forest Classifier

In [33]:
RF = RandomForestClassifier(featuresCol = 'features', labelCol= 'approved', maxDepth= 5, numTrees= 150)
RF_C = RF.fit(train_df)
RF_train = RF_C.transform(train_df)
RF_test = RF_C.transform(test_df)

#Model Evaluation
  

In [31]:
accuracyscore = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='accuracy')
accuracyscore.evaluate(RF_train)

1.0

In [32]:
accuracyscore.evaluate(RF_test)

1.0

In [34]:
Precision = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='weightedPrecision')
Precision.evaluate(RF_train)

1.0

In [35]:
F1Score = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='f1')
F1Score.evaluate(RF_train)

1.0

In [36]:
Recall = MulticlassClassificationEvaluator(predictionCol='prediction', labelCol='approved', metricName='weightedRecall')
Recall.evaluate(RF_train)

1.0

In [37]:
RF_train.groupBy('approved', 'prediction').count().show()

+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       0|       0.0|  135|
|       1|       1.0|  211|
+--------+----------+-----+



In [38]:
RF_test.groupBy('approved', 'prediction').count().show()


+--------+----------+-----+
|approved|prediction|count|
+--------+----------+-----+
|       0|       0.0|   68|
|       1|       1.0|   86|
+--------+----------+-----+

